# P5-4. 최종 미니프로젝트 — 딥러닝 모델링 — PyTorch 코드 구현 가이드

**주제: 고객 이탈(Churn) 예측 — PyTorch DNN 직접 구현**

## 핵심 구현 흐름
`데이터 준비 → Train/Test → Train/Validation → 스케일링 → TensorDataset → DataLoader → DNN → Loss/Optimizer → 학습/Validation → EarlyStopping → Test → pos_weight → 비교`

이 노트북은 완성 코드를 제공하지 않고, P4-6과 같이 단계별 요구사항을 보고 직접 구현하도록 구성한다.

## 0. 라이브러리와 실행 장치
PyTorch, NumPy, pandas, matplotlib, scikit-learn을 불러오고 CPU/GPU 장치를 설정한다.

In [1]:
# 필요한 라이브러리, RANDOM_STATE, device를 작성한다.

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')
 
# 시드 설정
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
 
# 장치 설정 (GPU 사용 가능하면 GPU, 아니면 CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

사용 장치: cuda


## 1. 데이터 로드
`data_save.csv`를 읽고 크기와 앞부분을 확인한다.

In [2]:
# 데이터를 불러온다.
df = pd.read_csv("data_v2_save.csv")
print(f"데이터셋 크기: {df.shape}")
print(f"\n첫 5행:\n{df.head()}")
print(f"\n데이터 타입:\n{df.dtypes}")
print(f"\n결측치:\n{df.isnull().sum()}")

데이터셋 크기: (7027, 17)

첫 5행:
   gender Partner Dependents  tenure     MultipleLines InternetService  \
0    Male      No         No      34                No             DSL   
1    Male      No         No       2                No             DSL   
2    Male      No         No      45  No phone service             DSL   
3  Female      No         No       2                No     Fiber optic   
4  Female      No         No       8               Yes     Fiber optic   

  OnlineSecurity OnlineBackup TechSupport StreamingTV StreamingMovies  \
0            Yes           No          No          No              No   
1            Yes          Yes          No          No              No   
2            Yes           No         Yes          No              No   
3             No           No          No          No              No   
4             No           No          No         Yes             Yes   

         Contract PaperlessBilling              PaymentMethod  MonthlyCharges  \
0       

## 2. 전처리
범주형 컬럼을 One-Hot Encoding하고 X와 y를 분리한다.

In [3]:
# 전처리를 수행한다.
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\n범주형 컬럼: {categorical_cols}")
 
# One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print(f"\nOne-Hot Encoding 후 크기: {df_encoded.shape}")
 
# X와 y 분리
y = df_encoded['Churn'].values
X = df_encoded.drop('Churn', axis=1).values
 
print(f"X 크기: {X.shape}, y 크기: {y.shape}")
print(f"Churn 분포:\n0: {(y==0).sum()}, 1: {(y==1).sum()}")


범주형 컬럼: ['gender', 'Partner', 'Dependents', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

One-Hot Encoding 후 크기: (7027, 27)
X 크기: (7027, 26), y 크기: (7027,)
Churn 분포:
0: 5161, 1: 1866


## 3. Train/Test 분할
P5-3과 동일하게 Test 30%, stratify, random_state=42를 사용한다.

In [ ]:
# Train 전체와 Test를 분리한다.
# X_train_full, y_train_full => 이후 학습/검증에 사용할 70% 데이터
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, 
    test_size=0.3, 
    stratify=y,  # 분류 문제에서 클래스 비율을 Train/Test에 비슷하게 유지
    random_state=42
)
 
print(f"\nTrain 전체: {X_train_full.shape}, Test: {X_test.shape}")


Train 전체: (4918, 26), Test: (2109, 26)


## 4. Train/Validation 분할
Train 전체에서 Validation 20%를 분리한다. Test는 마지막 평가까지 사용하지 않는다.

In [8]:
# Train과 Validation을 분리한다.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, 
    test_size=0.2, 
    stratify=y_train_full, 
    random_state=42
)
 
print(f"Train: {X_train.shape}, Validation: {X_val.shape}")

Train: (3934, 26), Validation: (984, 26)


## 5. 스케일링
MinMaxScaler를 Train에만 fit하고 Validation/Test에는 transform만 적용한다.

In [9]:
# 세 데이터셋을 스케일링하고 float32로 변환한다.
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)
 
print(f"\n스케일링 완료")
print(f"Train 범위: [{X_train_scaled.min():.3f}, {X_train_scaled.max():.3f}]")


스케일링 완료
Train 범위: [0.000, 1.000]


## 6. TensorDataset과 DataLoader
다음 조건으로 구성한다.
- X: `torch.float32`
- y: 이진분류이므로 `torch.float32`
- Train DataLoader: batch_size=32, shuffle=True
- Validation/Test: shuffle=False

In [10]:
# TensorDataset과 DataLoader를 구현한다.

# torch.Tensor로 변환
X_train_tensor = torch.from_numpy(X_train_scaled).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
 
X_val_tensor = torch.from_numpy(X_val_scaled).float()
y_val_tensor = torch.from_numpy(y_val).float().unsqueeze(1)
 
X_test_tensor = torch.from_numpy(X_test_scaled).float()
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)
 
# TensorDataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
 
# DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
 
print(f"Train DataLoader: {len(train_loader)} batches")
print(f"Validation DataLoader: {len(val_loader)} batches")
print(f"Test DataLoader: {len(test_loader)} batches")

Train DataLoader: 123 batches
Validation DataLoader: 31 batches
Test DataLoader: 66 batches


## 7. DNN 모델 정의
`nn.Module`을 상속하여 다음 구조를 만든다.

`Linear(input,64) → ReLU → Dropout(0.3) → Linear(64,32) → ReLU → Dropout(0.3) → Linear(32,1)`

**주의:** 마지막 Sigmoid는 모델 안에 넣지 않는다.

In [11]:
# ChurnNet 클래스를 구현하고 device로 이동한다.

class ChurnNet(nn.Module):
    def __init__(self, input_size):
        super(ChurnNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)
        
        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        
        self.fc3 = nn.Linear(32, 1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        
        x = self.fc3(x)
        return x
 
# 모델 인스턴스 생성
input_size = X_train_scaled.shape[1]
model = ChurnNet(input_size).to(device)
print(f"\n모델 구조:\n{model}")
print(f"총 파라미터: {sum(p.numel() for p in model.parameters())}")


모델 구조:
ChurnNet(
  (fc1): Linear(in_features=26, out_features=64, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
)
총 파라미터: 3841


## 8. Loss와 Optimizer
기본 모델은 `BCEWithLogitsLoss`, Optimizer는 Adam(lr=0.001)을 사용한다.

In [12]:
# criterion과 optimizer를 정의한다.

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
 
print(f"\nLoss Function: BCEWithLogitsLoss")
print(f"Optimizer: Adam (lr=0.001)")


Loss Function: BCEWithLogitsLoss
Optimizer: Adam (lr=0.001)


## 9. 학습 + Validation
직접 학습 루프를 구현한다.

Train 순서:
`model.train() → forward → loss → zero_grad → backward → step`

Validation:
`model.eval() → torch.no_grad()`

매 epoch loss와 accuracy를 저장한다.

In [ ]:
# train_model() 함수를 구현한다.

def train_model(model, train_loader, val_loader, criterion, optimizer, 
                epochs=100, patience=5, device='cpu'):
    """
    학습 함수 with EarlyStopping
    
    Parameters:
    - model: 학습할 모델
    - train_loader: 학습 데이터로더
    - val_loader: 검증 데이터로더
    - criterion: 손실함수
    - optimizer: 최적화기
    - epochs: 최대 에포크
    - patience: 조기종료 인내 기간
    - device: 실행 장치
    
    Returns:
    - history: {'train_loss', 'train_acc', 'val_loss', 'val_acc'} 저장
    """
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # ===== Train =====
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            # Forward
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # 통계
            train_loss += loss.item() * X_batch.size(0)
            
            # Accuracy 계산 (threshold=0.5)
            preds = (torch.sigmoid(logits) > 0.5).float()
            train_correct += (preds == y_batch).sum().item()
            train_total += y_batch.size(0)
        
        train_loss_avg = train_loss / train_total
        train_acc = train_correct / train_total
        history['train_loss'].append(train_loss_avg)
        history['train_acc'].append(train_acc)
        
        # ===== Validation =====
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                
                val_loss += loss.item() * X_batch.size(0)
                preds = (torch.sigmoid(logits) > 0.5).float()
                val_correct += (preds == y_batch).sum().item()
                val_total += y_batch.size(0)
        
        val_loss_avg = val_loss / val_total
        val_acc = val_correct / val_total
        history['val_loss'].append(val_loss_avg)
        history['val_acc'].append(val_acc)

## 10. EarlyStopping
Validation loss가 좋아질 때 모델 가중치를 저장하고, 5 epoch 동안 개선이 없으면 종료한다.
학습 종료 후 가장 좋은 가중치를 다시 불러온다.

In [ ]:
# train_model() 내부에 EarlyStopping 로직을 추가한다.

## 11. 학습곡선
Train/Validation loss와 accuracy를 그래프로 비교한다.

In [ ]:
# history를 이용해 그래프를 그린다.

## 12. Test 평가 함수
예측 흐름은 다음과 같다.

`logits → torch.sigmoid() → threshold 0.5 → 0/1`

Accuracy, Precision, Recall, F1을 계산한다.

In [ ]:
# predict_binary()와 evaluate_dnn()을 구현한다.

## 13. 기본 DNN 최종 평가
학습 중 사용하지 않은 Test 데이터로 baseline 모델을 평가한다.

In [ ]:
# baseline 성능과 classification report를 출력한다.

## 14. 클래스 불균형 대응
`compute_class_weight()`로 `w0`, `w1`을 계산하고 다음 값을 만든다.

`pos_weight = w1 / w0`

새 모델의 `BCEWithLogitsLoss(pos_weight=...)`에 적용한다.

In [ ]:
# pos_weight를 계산한다.

## 15. weighted DNN 재학습 및 평가
같은 구조의 새 모델을 만들고 weighted loss로 다시 학습한다. baseline과 Test 성능을 비교한다.

In [ ]:
# weighted 모델을 학습하고 평가한다.

## 16. 최종 점검표
| 확인 항목 | 핵심 기준 |
|---|---|
| 데이터 분리 | P5-3과 동일한 Test 조건인가 |
| Validation | Train에서 별도로 만들었는가 |
| TensorDataset | X/y dtype이 적절한가 |
| DataLoader | Train만 shuffle=True인가 |
| 모델 | 최종 출력 1개 logit인가 |
| Loss | BCEWithLogitsLoss인가 |
| 학습 | zero_grad → backward → step이 있는가 |
| 평가 | eval + no_grad를 사용했는가 |
| EarlyStopping | best weight를 복원했는가 |
| 불균형 | pos_weight 전후를 비교했는가 |
| 최종 평가 | Accuracy, Precision, Recall, F1을 비교했는가 |